# PG-LIF -- P1 Paper-Mode: v9 (skip-on-nonfinite-gradient fix)

**v9: root cause of the PG-LIF-family training collapses confirmed, and the real fix applied.** A dedicated
collapse-diagnostic run (targeting plain PGLIF, the exact config+seed that collapsed in v8) caught the event
directly: at epoch 13, the forward pass and loss stayed perfectly finite (loss=2.97), and every clamped
state variable was exactly at or safely under its bound (max|vd|=max|vs|=20.0 = V_CLAMP exactly; max|p|=11.1,
well under P_CLAMP=50) -- yet the BACKWARD pass produced a non-finite gradient confined entirely to
`layer1`'s parameters (`w_in`, `w_rec`, `ap_logit`, `kappa`). This is a classic BPTT exploding-gradient event
through `layer1`'s Jacobian over 250 unrolled timesteps (well-characterized in the RNN literature, e.g.
Pascanu et al. 2013) -- NOT a magnitude runaway, which v8's clamps could never have prevented since
`torch.clamp` does not repair NaN (IEEE-754: any comparison with NaN is false, so NaN passes through clamp
unchanged).

**Why v8 failed despite this.** The codebase already had a skip-on-inf/nan mechanism -- but it lived entirely
inside `GradScaler.step()`, which is a transparent no-op passthrough to `opt.step()` when `USE_AMP=False`.
Since AMP is disabled by design (it gave negligible speedup, see the AMP history below), v8's actual training
runs had **zero protection** against this event: the corrupted gradient was applied to the weights every
time it occurred. v9 adds an explicit, AMP-independent finiteness check after every `backward()`: if any
parameter's gradient is non-finite, that one optimizer step is discarded (gradients zeroed, `opt.step()`
never called) and training continues to the next batch. This is the standard, well-precedented response to
BPTT gradient explosion -- detect and discard the corrupted update, rather than try to prevent the underlying
instability from ever occurring. v8's forward-pass clamps (`V_CLAMP`, `P_CLAMP`) are KEPT, since they are
confirmed to correctly bound the forward pass (a separate, real safeguard) even though they were never the
mechanism protecting against this specific backward-pass failure.

**Skip-rate reporting.** Every run now records `total_skipped_steps` / `total_steps` / `skip_rate` in its
result JSON, and prints a per-epoch warning if any steps were skipped -- escalating to a SEVERE warning if
skip rate exceeds 30% in an epoch (a sign the run is fundamentally unstable, not experiencing rare events).
This is intended to be reported plainly in the manuscript's reproduction notes: "N of M optimizer steps were
skipped due to non-finite gradients" is honest methodology, not a result to hide.

**Config unchanged otherwise.** v3's training protocol (LR=5e-4, `MultiStepLR([40,80], gamma=0.1)`,
dropout=0.1, Adam) combined with v6's architectural corrections (recurrent weight `bias=True`/default init,
official spike binning) and v8's forward-pass clamps remain the base. `FIX_VERSION=9`: only PG-LIF and the
ALIF2 ablation are affected (their training LOOP behavior changed); the four baselines are reused from
`paper_mode_final`/`paper_mode_v8_clamped` automatically -- no wasted compute. Results write to a fresh
`paper_mode_v9/` folder.

**Models trained (T=250, 100 epochs, drops at [40,80], 9 tags):** LIF, ALIF, TC-LIF, DH-LIF, PG-LIF, and four
ablations (plateau->2nd-ALIF-variable, kappa=0, theta_d jitter, no refractory). Ablation (c) [gate removed]
is a Regime B concept and is not run here (the gate has no effect on offline BPTT). Runtime: resumable via
per-(tag,seed) checkpointing; only the 5 PG-LIF-family tags need to actually train (4 baselines reused).

In [ ]:
import os, json, time, math
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/My Drive'
    if not os.path.isdir(ROOT): ROOT = '/content/drive/MyDrive'
    if not os.path.isdir(ROOT):
        raise RuntimeError("Drive mounted but neither 'My Drive' nor 'MyDrive' exists under /content/drive - "
                            "the mount likely did not complete. Re-run this cell and approve the Drive access prompt.")
    BASE = os.path.join(ROOT, 'PG_LIF')
    IN_COLAB = True
except ImportError:
    print('WARNING: not running in Colab (google.colab not importable) - using a local, non-persistent '
          'folder. Results and downloaded data will NOT survive a session restart.')
    BASE = './PG_LIF'
except Exception as e:
    raise RuntimeError(
        f'Google Drive did not mount correctly ({e}). Results MUST live on Drive to persist across '
        'sessions and to find your existing PG_LIF/data/SHD cache and prior run folders. Fix the mount '
        '(re-run this cell, approve the access prompt) before continuing - do NOT proceed on a silent '
        'local fallback, or you will re-download data and lose access to all previous results.') from e
DATA = os.path.join(BASE, 'data', 'SHD')
OUT_ROOT = os.path.join(BASE, 'P1_results')
os.makedirs(DATA, exist_ok=True)
print('Running in Colab, Drive mounted.' if IN_COLAB else 'Running locally (dev/test only).')
print('Base folder:', BASE)
print('Results root:', OUT_ROOT)

In [ ]:
# --- CONFIG: adjust per session; already-finished (tag, seed) pairs are skipped automatically ---
SANITY_CHECK = False           # Two-layer architecture check (already passed: TC-LIF climbed 27%->75%
                                # by ep14 on a compressed schedule). Leave False unless re-diagnosing the
                                # architecture itself.
VERIFY_COMPILE_FIX = False      # (kept for reference) cheap LIF-only check that confirmed the
                                # torch.compile stale-graph fix. Already passed -- leave False.
VALIDATE_READOUT_FIX = False    # (kept for reference) short 40-epoch TC-LIF check that confirmed the
                                # readout fix: TC-LIF reached 80.65% by ep15 (vs. ~50 epochs pre-fix) and
                                # was still climbing at 82.77% when the shortened schedule ended. That
                                # confirmed the fix but left a ~6pp gap to the published 88.91% -- likely
                                # because the validation schedule was truncated (40 epochs, one LR drop)
                                # vs. the real protocol (100 epochs, drops at 40 and 80). Hence the mode below.
VALIDATE_FULL_PROTOCOL = False  # STOPPED validating individual hyperparameters. v3's config (readout fix,
                                # LR=5e-4, MultiStepLR[40,80], dropout=0.1, Adam) is the one setting with a
                                # confirmed, verified result (85.20% TC-LIF) and is now final. Moving to the
                                # real 9-tag, multi-seed comparative sweep -- that is the actual paper need,
                                # not an exact match to TC-LIF's published number. Set True only if you
                                # specifically want to re-run this single-model validation again.

if SANITY_CHECK:
    SEEDS = [0]
    MODELS_TABLE1 = ['TCLIF', 'DHLIF', 'PGLIF']   # TC-LIF and DH-LIF have published targets; plus PG-LIF
    MODELS_TABLE3 = []
    EPOCHS_OVERRIDE = 15
elif VALIDATE_FULL_PROTOCOL:
    SEEDS = [0]
    MODELS_TABLE1 = ['TCLIF']   # TC-LIF under the real 100-epoch protocol, banked to the real folder
    MODELS_TABLE3 = []
    EPOCHS_OVERRIDE = None       # None -> use the full EPOCHS=100 with the official StepLR(10,0.5) below
elif VALIDATE_READOUT_FIX:
    SEEDS = [0]
    MODELS_TABLE1 = ['TCLIF']   # the one model with a firm published target (88.91%) to validate against
    MODELS_TABLE3 = []
    EPOCHS_OVERRIDE = 40        # pre-fix TC-LIF was at ~75% by ep15 and stalled ~82%; 40 epochs is
                                 # plenty to see whether it now clearly exceeds that
elif VERIFY_COMPILE_FIX:
    SEEDS = [0]
    MODELS_TABLE1 = ['LIF']    # LIF gave the clearest evidence of the compile bug -- fastest, cheapest check
    MODELS_TABLE3 = []
    EPOCHS_OVERRIDE = 25       # the original bug appeared within the first 11-14 epochs, so 25 is a safe margin
else:
    SEEDS = [0, 1, 2, 3, 4]         # v10: widened from [0,1,2] to 5 seeds for higher statistical power
                                     # (reviewer-requested); seeds 0-2 are already finished for every existing
                                     # tag and will be [skip]ped via the resume logic, only 3-4 train fresh.
                                     # The new ablF tag has no finished seeds yet and trains all 5 fresh.
    MODELS_TABLE1 = ['LIF', 'ALIF', 'TCLIF', 'DHLIF', 'PGLIF']
    MODELS_TABLE3 = ['PGLIF_ablA_alif2', 'PGLIF_ablB_kappa0', 'PGLIF_ablD_thetajitter', 'PGLIF_ablE_norefrac',
                      'PGLIF_ablF_dendriteonly']   # v10: new ablation, see TAG_SPEC below
    EPOCHS_OVERRIDE = None
MODELS_TABLE1 = list(MODELS_TABLE1); MODELS_TABLE3 = list(MODELS_TABLE3)
RUN_THIS_SESSION = MODELS_TABLE1 + MODELS_TABLE3   # comment out models you don't want to (re)run this session

T_BINS, EPOCHS = 250, 100        # official TC-LIF SHD protocol. T_BINS stays 250 in every mode, including
                                  # short validation passes: the published numbers are for this length.
if EPOCHS_OVERRIDE:
    EPOCHS = EPOCHS_OVERRIDE
# Scheduler: StepLR(step_size=10, gamma=...). The repo's main.py (line 309) hard-codes StepLR(10, 0.5),
# overriding the --schedule [40,80] argument default this notebook originally copied -- but the paper's
# own methods section (confirmed identically in two independent papers by the same authors) states the
# SHD schedule actually used is gamma=0.8, not 0.5: "decaying to 0.8 times its previous value after every
# 10 epochs." Applying gamma=0.5 on top of an already-too-low LR (the v4 mistake) made things worse by
# decaying an insufficient rate even faster; the real fix (v5) is BOTH the correct LR (see BATCH/LR above)
# AND gamma=0.8, not gamma=0.5 alone.
# Scheduler reverted to v3's confirmed setting: MultiStepLR(milestones=[40,80], gamma=0.1). The StepLR(10,
# 0.8/0.5) variants (v4/v5/v6) were paper-text-motivated but never confirmed to actually beat this in a
# real, verified run. Stopping the hyperparameter search here.
SCHEDULE = [max(1, int(EPOCHS * 0.4)), max(2, int(EPOCHS * 0.8))]   # scales to [40,80] at EPOCHS=100
WEIGHT_DECAY = 0.0              # v3 used Adam with no weight decay; kept for consistency.
EARLY_STOP_MIN_EPOCH = min(SCHEDULE) + 5   # reverted to v3's exact condition: arm 5 epochs after the
                                            # first LR drop (ep45 at the full 100-epoch protocol). This is
                                            # the condition that produced the confirmed 85.20% TC-LIF result.
EARLY_STOP_PATIENCE = 15   # stop a (tag, seed) run if test acc hasn't improved for this many epochs
                            # once past EARLY_STOP_MIN_EPOCH; set to None to disable and always run EPOCHS
# Kept for optional future use: if a VALIDATE_* mode is ever re-enabled, disable early stopping so the
# full unmodified curve is visible (not needed for the default full-sweep mode below).
if VALIDATE_FULL_PROTOCOL or VALIDATE_READOUT_FIX:
    EARLY_STOP_PATIENCE = None
DROPOUT = 0.1               # reverted to v3's confirmed value (85.20% TC-LIF). The official-default 0.0
                            # (v4-v6) was correctness-motivated but never confirmed to help; stopping here.
USE_RECURRENCE = True       # KEEP TRUE. The repo's --network flag defaults to 'ff' (feedforward), but the
                            # parameter count settles it: with recurrence ON this model has 141,592 params,
                            # matching the paper's stated 141.8K almost exactly; with it OFF, 108,824 -- far
                            # from the published figure. So the 88.91% result was produced by a recurrent
                            # variant (the repo ships several, e.g. fb_SHD_v1), and the parameter match is
                            # meaningful evidence, not a coincidence. Set False only to deliberately test the
                            # pure-feedforward configuration; it will NOT match the published parameter count.
V_CLAMP = 20.0              # membrane clamp (|vs|,|vd| <= V_CLAMP each step). Numerical safeguard against the
                            # positive-feedback runaway that collapsed PG-LIF-family runs in the 3-seed sweep
                            # (plateau/a2 ADD to the membrane -> more spikes -> more drive -> NaN death; TC-LIF
                            # /ALIF are immune since their slow variable RAISES the threshold instead).
                            # Healthy runs measured at |v|~1-9, so a +-20 clamp never activates for stable
                            # runs -- it only caps a divergence before it hits NaN. Standard in mature SNN
                            # libraries (snnTorch, SpikingJelly). Applies only to PG-LIF and the ALIF2 ablation.
P_CLAMP = 50.0              # plateau/a2 state clamp. Manuscript Prop. 2 bounds the plateau at P0/(1-ap^nref),
                            # which DIVERGES as the refractory nref->0 (ablation E, which collapsed 3/3); this
                            # makes the bound explicit and finite for every configuration. Well above the
                            # healthy operating range (p<=~9 measured), so it too never bites stable runs.
USE_AMP = False             # DISABLED after diagnosing a likely silent-training-failure mode: fp16's max
                            # representable value (~65504) is far below the raw gradient magnitudes measured
                            # for this architecture (up to ~1e7-1e8 for plain LIF over T=250 in float32).
                            # Under autocast, this plausibly overflows to inf/nan in the FORWARD pass itself;
                            # GradScaler then silently skips the optimizer step with NO error or warning --
                            # training "runs" (normal timing, no exceptions) while weights never update. This
                            # matches the exact-frozen spk/sample values observed across dozens of epochs in
                            # the last real run (LIF, ALIF, DHLIF, PGLIF, and ablations all flatlined this
                            # way). AMP also measured to give only a small (~10%) speed gain here anyway
                            # (see USE_COMPILE below), so the trade-off clearly favors leaving it off. A
                            # skip-counter is added below so if you ever re-enable this, the failure mode
                            # becomes visible in the printed log instead of silent.
USE_COMPILE = False         # DISABLED after diagnosing a serious correctness bug, not just a missed
                            # speedup. Evidence from a completed 3-seed sweep: spk/sample was BIT-IDENTICAL
                            # from epoch 0 for 11+ consecutive epochs (e.g. LIF: 575.70 unchanged, ep00-ep10)
                            # while test accuracy clearly improved (3.9% -> 22.6%) over the same epochs --
                            # only possible if the spike-generating computation (through w_in/w_rec) was
                            # reading a STALE, first-compile snapshot while only w_out genuinely updated.
                            # This is a known torch.compile/Dynamo failure mode when a custom autograd.Function
                            # (our Triangle spike surrogate) is combined with a stateful module that reassigns
                            # plain Python attributes (self.v, self.a, self.p -- never registered as buffers)
                            # across calls: the graph can silently fail to re-trace and reuse a stale snapshot.
                            # TC-LIF's healthy result in that same run is NOT counter-evidence: it was reused
                            # from an earlier session and was never actually retrained under compile there.
                            # Re-enable only after restructuring the neuron cells to use registered buffers
                            # and verifying weights genuinely change every epoch on a short real-GPU test.
OUT = os.path.join(OUT_ROOT, 'sanity_check_2layer' if SANITY_CHECK else
                    'validate_readout_fix' if VALIDATE_READOUT_FIX else
                    'verify_compile_fix' if VERIFY_COMPILE_FIX else 'paper_mode_v9')
# v8 writes to its OWN folder (paper_mode_v9), SEPARATE from the v7 'paper_mode_final' results.
# The four baselines (LIF/ALIF/TCLIF/DHLIF) are byte-identical in v8, so the cross-folder reuse logic in
# train_one() copies their finished v7 results from paper_mode_final into this folder automatically (no
# retraining); only the five PG-LIF-family tags, whose cells gained the clamp, train fresh here. Keeping
# v7 and v8 in separate folders means the pre-clamp results stay intact for comparison and nothing is
# silently overwritten. (VALIDATE_FULL_PROTOCOL, if ever re-enabled, shares this folder deliberately.)
os.makedirs(OUT, exist_ok=True)
print('This run\'s results folder:', OUT)

# --- realistic runtime estimate (observed: ~72-104s/epoch on a T4 at T=250, HIDDEN=128) ---
SEC_PER_EPOCH = 90
n_runs = len(RUN_THIS_SESSION) * len(SEEDS)
worst_case_hours = n_runs * EPOCHS * SEC_PER_EPOCH / 3600
print(f'\nThis session queues {n_runs} (tag, seed) runs x up to {EPOCHS} epochs each.')
if EARLY_STOP_PATIENCE is None:
    print(f'Worst case: ~{worst_case_hours:.1f} GPU-hours. Early stopping is DISABLED this run -- every '
          f'(tag, seed) runs the full {EPOCHS} epochs (this is deliberate for validation runs, so we see '
          f'the true, unmodified curve rather than one truncated by a heuristic the paper does not use).')
else:
    print(f'Worst case (no early stop): ~{worst_case_hours:.1f} GPU-hours. Early stopping (patience='
          f'{EARLY_STOP_PATIENCE}) will cut this substantially for models that plateau early.')
print('Colab sessions time out well before this for a full 9x5 sweep - budget several sessions,')
print('narrow RUN_THIS_SESSION / SEEDS per session, and rely on the per-(tag,seed) resume/skip logic.')
BATCH, LR, HIDDEN, MAX_TIME = 64, 5e-4, 128, 1.4   # LR reverted to v3's confirmed value (85.20% TC-LIF).
# The 5e-3 "paper-text" LR (v5/v6) was never actually confirmed working -- stopping the hyperparameter
# chase here and using the one setting with a real, verified result.
# LR corrected per the paper's own methods section (confirmed independently in TWO papers by the same
# authors: arXiv:2308.13250 and the companion arXiv:2307.07231 LSTM-LIF paper, identical wording): "The
# initial learning rate is set to 0.0005, and 0.005 for feedforward and recurrent networks on the SHD
# dataset". We run the RECURRENT network (USE_RECURRENCE=True, confirmed by the 141.8K parameter match),
# so LR should be 0.005, NOT 0.0005 -- an earlier version used the feedforward value for a recurrent
# network, a 10x LR shortfall that plausibly explains every previous run undershooting the published
# 88.91% in the same shape: real learning, consistently short, as if never given enough effective step
# size to fully converge in 100 epochs. NOTE: the repo's own SHD "Reproduce" command does not pass an
# explicit --lr flag (which would default to 0.0005), so there is a minor inconsistency between the
# abbreviated repo snippet and the paper's stated methodology; we follow the paper text, confirmed twice.
N_IN, N_OUT = 700, 20
print('This session will train/resume:', RUN_THIS_SESSION, '× seeds', SEEDS)

## Data (downloads once into the Drive cache if not already present, then reused)

In [ ]:
import numpy as np, h5py, torch, torch.nn as nn, gzip, shutil, urllib.request
URLS = {'shd_train.h5': 'https://zenkelab.org/datasets/shd_train.h5.gz',
        'shd_test.h5':  'https://zenkelab.org/datasets/shd_test.h5.gz'}
for name, url in URLS.items():
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst):
        gz = dst + '.gz'
        print('downloading', url, '...')
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as fi, open(dst, 'wb') as fo: shutil.copyfileobj(fi, fo)
        os.remove(gz)
    print(name, 'ready,', os.path.getsize(dst) // (1 << 20), 'MB, at', dst)

def load_split(fname):
    with h5py.File(os.path.join(DATA, fname), 'r') as f:
        return ([np.array(t) for t in f['spikes']['times']],
                [np.array(u) for u in f['spikes']['units']],
                np.array(f['labels'], dtype=np.int64))
TR_raw = load_split('shd_train.h5'); TE_raw = load_split('shd_test.h5')

def precompute_dense(split, name):
    """Bin every sample ONCE into a dense (N, T_BINS, N_IN) bool tensor. The previous version rebuilt
       this per sample on every single batch of every single epoch (measured: ~1.4x needless CPU cost,
       and worse, blocks the GPU pipeline every batch) -- fixed by paying this cost exactly once.

       v6: binning corrected to match the official SpikeIterator exactly: `np.digitize(firing_times,
       np.linspace(0, max_time, num=nb_steps))`. This is NOT the same as a naive equal-partition of
       [0, max_time) into T_BINS bins (which is what an earlier version did): linspace places T_BINS
       POINTS spanning [0, max_time] inclusive, and digitize returns index i such that
       bins[i-1] <= t < bins[i] -- so t=0 lands in index 1, not 0, meaning bin 0 is permanently empty
       for real data and the network effectively gets one wasted leading timestep before any input
       arrives (249 informative bins out of 250, not 250). We reproduce this exactly for fidelity,
       rather than the more efficient-looking scheme, since the goal here is matching the reference,
       not improving on it. One safety deviation: digitize can return T_BINS itself when t >= max_time
       exactly (out of bounds for a T_BINS-wide tensor); we clip that single edge case rather than
       crash, since SHD's real spike times are not expected to land exactly on the boundary."""
    times, units, labels = split
    n = len(labels)
    time_bins = np.linspace(0, MAX_TIME, num=T_BINS)
    X = torch.zeros(n, T_BINS, N_IN, dtype=torch.bool)
    for i in range(n):
        tt = times[i]; uu = units[i]
        tb = np.clip(np.digitize(tt, time_bins), 0, T_BINS - 1)
        X[i, tb, uu] = True
    print(f'{name}: precomputed {n} samples, {X.element_size()*X.nelement()/1e9:.2f} GB (bool)')
    return X, torch.as_tensor(labels, dtype=torch.long)

TR_X, TR_Y = precompute_dense(TR_raw, 'train')
TE_X, TE_Y = precompute_dense(TE_raw, 'test')

def batches(X, Y, batch_size, shuffle, device='cpu', drop_last=False):
    n = len(Y)
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    if drop_last: n = (n // batch_size) * batch_size   # keep batch shape static -> fewer torch.compile recompiles
    for b0 in range(0, n, batch_size):
        sel = idx[b0:b0 + batch_size]
        yield X[sel].float().to(device), Y[sel].to(device)
print('train', len(TR_Y), '| test', len(TE_Y))

## Neuron cells
Baselines (LIF, ALIF, official TC-LIF dynamics, DH-LIF re-implementation) and PG-LIF (scaled configuration, the P1 consolidation winner) are carried over unchanged from the validated P1 v2 harness. Three new cells implement ablations (a), (d), (e); ablation (b) reuses PGLIFCell with κ fixed to 0.

In [ ]:
class Triangle(torch.autograd.Function):
    gamma = 1.0
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x); return (x >= 0).float()
    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * torch.clamp(1.0 - x.abs() / Triangle.gamma, min=0.0)
spike_fn = Triangle.apply
def decay(tau): return math.exp(-1.0 / tau)

class LIFCell(nn.Module):
    th = 1.0
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20)
    def init(self, B, dev): self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th
        return s

class ALIFCell(nn.Module):
    th = 1.0; beta = 1.6
    def __init__(self, N): super().__init__(); self.N = N; self.am = decay(20); self.aa = decay(200)
    def init(self, B, dev):
        self.v = torch.zeros(B, self.N, device=dev); self.a = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v = self.am * self.v + I
        th = self.th + self.beta * self.a
        s = spike_fn(self.v - th)
        self.v = self.v - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class TCLIFCell(nn.Module):
    """Official TC-LIF dynamics (ZhangShimin1/TC-LIF)."""
    th = 1.5; gamma_r = 0.5
    def __init__(self, N):
        super().__init__(); self.N = N; self.d = nn.Parameter(torch.zeros(2))
    def init(self, B, dev):
        self.v1 = torch.zeros(B, self.N, device=dev); self.v2 = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        self.v1 = self.v1 - torch.sigmoid(self.d[0]) * self.v2 + I
        self.v2 = self.v2 + torch.sigmoid(self.d[1]) * self.v1
        s = spike_fn(self.v2 - self.th)
        self.v1 = self.v1 - s * self.gamma_r
        self.v2 = self.v2 - s * self.th
        return s

class DHLIFCell(nn.Module):
    """Re-implementation per Zheng et al. 2024. FIXED: the branch update now includes the (1-ad)
       normalization; without it (as originally shipped), each branch's steady-state gain to a sustained
       input scales as 1/(K*(1-ad)), giving the slowest branch (ad=0.99) a ~50x larger gain than the
       fastest (ad=0.5) -- exactly the leak-amplification pattern already diagnosed for PG-LIF's plateau,
       here in DH-LIF's dendritic branches. Confirmed as the cause of an anomalous 12,861 spikes/sample
       already at epoch 0 (vs. 50-2000 for every other model) in a completed run, with accuracy stuck at
       ~20% versus the ~90% published for DH-SNN."""
    th = 1.0; K = 4
    def __init__(self, N):
        super().__init__(); self.N = N; self.am = decay(20)
        init_a = torch.tensor([0.5, 0.8, 0.95, 0.99])
        logit = torch.log(init_a / (1 - init_a))
        self.branch_logit = nn.Parameter(logit.view(self.K, 1).repeat(1, N))
        self.mix = nn.Parameter(torch.ones(self.K, N) / self.K)
    def init(self, B, dev):
        self.i = torch.zeros(B, self.K, self.N, device=dev); self.v = torch.zeros(B, self.N, device=dev)
    def forward(self, I):
        ad = torch.sigmoid(self.branch_logit)
        self.i = ad.unsqueeze(0) * self.i + (1 - ad).unsqueeze(0) * (I.unsqueeze(1) / self.K)
        self.v = self.am * self.v + (self.mix.unsqueeze(0) * self.i).sum(1)
        s = spike_fn(self.v - self.th)
        self.v = self.v - s.detach() * self.th
        return s

class PGLIFCell(nn.Module):
    """PG-LIF, scaled configuration (P1 consolidation winner): drive kappa*p*(1-alpha_m).
       theta_d and tref_p are exposed for ablations (d) and (e); kappa_fixed_zero for ablation (b)."""
    th = 1.0; P0 = 1.0; beta = 1.0
    def __init__(self, N, theta_d=1.0, theta_d_jitter=0.0, tref_p=10, kappa_fixed_zero=False, dendrite_only=False):
        super().__init__(); self.N = N
        self.am = decay(20); self.ad = decay(20); self.aa = decay(200)
        ap0 = decay(T_BINS / 2)
        self.ap_logit = nn.Parameter(torch.full((N,), math.log(ap0 / (1 - ap0))))
        self.kzero = kappa_fixed_zero
        self.kappa = nn.Parameter(torch.zeros(N)) if kappa_fixed_zero else nn.Parameter(torch.ones(N))
        self.tref_p = tref_p
        self.dendrite_only = dendrite_only   # ablation: I_ff routes to the dendrite ONLY; the soma receives
                                              # no direct feedforward term, only recurrent input + the plateau
                                              # drive. Tests the manuscript's claim (Section 6.2) that forcing
                                              # all feedforward information through the all-or-none plateau
                                              # acts as an information bottleneck that collapses accuracy.
        td = torch.full((N,), float(theta_d))
        if theta_d_jitter > 0:
            td = td * torch.empty(N).uniform_(1 - theta_d_jitter, 1 + theta_d_jitter)
        self.register_buffer('theta_d', td)
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.vd, self.p, self.a = z(), z(), z(), z()
        self.rp = torch.zeros(B, self.N, device=dev)
    def forward(self, I_ff, I_rec):
        self.vd = self.ad * self.vd + I_ff
        self.vd = torch.clamp(self.vd, -V_CLAMP, V_CLAMP)   # v6-stability: numerical safeguard against the
                                                             # positive-feedback runaway diagnosed in the
                                                             # 3-seed sweep (PG-LIF family collapsed 1-3/3);
                                                             # healthy dynamics sit at |v|~1-9, well inside
                                                             # +-V_CLAMP, so this never activates for stable
                                                             # runs and only caps a divergence before NaN.
        ed = spike_fn(self.vd - self.theta_d) * (self.rp == 0).float()
        self.rp = torch.clamp(self.rp - 1, min=0) + ed.detach() * self.tref_p
        self.p = torch.sigmoid(self.ap_logit) * self.p + self.P0 * ed
        self.p = torch.clamp(self.p, max=P_CLAMP)           # plateau state bound (Prop. 2 diverges as the
                                                             # refractory -> 0, i.e. ablation E; this makes
                                                             # the bound explicit and finite for all configs)
        drive = 0.0 if self.kzero else self.kappa * self.p * (1 - self.am)
        ff_to_soma = 0.0 if self.dendrite_only else I_ff
        self.vs = self.am * self.vs + ff_to_soma + I_rec + drive
        self.vs = torch.clamp(self.vs, -V_CLAMP, V_CLAMP)   # same safeguard on the soma
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        return s

class ALIF2Cell(nn.Module):
    """Ablation (a): the plateau/dendrite pathway is replaced by a SECOND independent adaptation
       variable a2 of the same time constant as the plateau (tau_p), driving the soma additively
       exactly where kappa*p sat in PGLIFCell, with the same feedforward routing. If this matches
       PGLIFCell's accuracy, the event-triggered/all-or-none character is not doing the work."""
    th = 1.0; beta = 1.0; beta2 = 1.0
    def __init__(self, N):
        super().__init__(); self.N = N
        self.am = decay(20); self.aa = decay(200)
        self.a2_decay = decay(T_BINS / 2)   # same time constant as the plateau (tau_p)
    def init(self, B, dev):
        z = lambda: torch.zeros(B, self.N, device=dev)
        self.vs, self.a, self.a2 = z(), z(), z()
    def forward(self, I_ff, I_rec):
        self.a2 = torch.clamp(self.a2, max=P_CLAMP)          # a2 adds to the membrane (positive feedback,
                                                              # same runaway mode as PG-LIF's plateau); bound it
        self.vs = self.am * self.vs + I_ff + I_rec + self.beta2 * self.a2 * (1 - self.am)
        self.vs = torch.clamp(self.vs, -V_CLAMP, V_CLAMP)    # same numerical safeguard as PGLIFCell
        th = self.th + self.beta * self.a
        s = spike_fn(self.vs - th)
        self.vs = self.vs - s.detach() * th.detach()
        self.a = self.aa * self.a + s.detach()
        self.a2 = self.a2_decay * self.a2 + s.detach()   # second slow variable, spike-triggered like ALIF
        return s

class RecLayer(nn.Module):
    """One recurrent spiking layer: input -> this layer's neurons -> recurrent self-connection."""
    def __init__(self, cell_name, n_in, n_hidden, **cell_kw):
        super().__init__()
        base = cell_name.split('_')[0] if '_' in cell_name else cell_name
        ctor = {'LIF': LIFCell, 'ALIF': ALIFCell, 'TCLIF': TCLIFCell, 'DHLIF': DHLIFCell,
                'PGLIF': PGLIFCell, 'ALIF2': ALIF2Cell}[base]
        self.w_in = nn.Linear(n_in, n_hidden)
        self.drop = nn.Dropout(DROPOUT)     # official ff_SHD applies Dropout after each linear layer;
                                             # SHD is small (8156 train samples) and overfits without it
        self.use_rec = USE_RECURRENCE
        if self.use_rec:
            # v6: matches official ElementWiseRecurrentContainer exactly: nn.Linear(hid_dim, hid_dim) with
            # default bias=True and DEFAULT init. An earlier version used bias=False and orthogonal init on
            # this weight; the official code's own orthogonal-init line for this exact weight is present
            # in source but COMMENTED OUT (`#nn.init.orthogonal_(self.hid_weight.weight)`), confirming the
            # actual reference model never applies it -- both were previously-unchecked deviations.
            self.w_rec = nn.Linear(n_hidden, n_hidden, bias=True)
        self.cell = ctor(n_hidden, **cell_kw)
        self.dual = base in ('PGLIF', 'ALIF2')
        self.n_hidden = n_hidden
    def init(self, B, dev):
        self.cell.init(B, dev)
        self.s = torch.zeros(B, self.n_hidden, device=dev)
    def step(self, x_t):
        iff = self.drop(self.w_in(x_t))
        irec = self.w_rec(self.s) if self.use_rec else torch.zeros_like(iff)
        self.s = self.cell(iff, irec) if self.dual else self.cell(iff + irec)
        return self.s

class RecSNN(nn.Module):
    """Two stacked spiking layers, 700-128-128-20, matching the official TC-LIF/DH-SNN SHD protocol
       (Zhang et al. 2024; Zheng et al. 2024).

       READOUT FIXED (v3): the official ff_SHD applies a plain nn.Linear(hidden, out_dim) to each
       timestep's spikes and sums those projections directly over time (res.sum(0)). An earlier version
       of this notebook instead ran the readout through a leaky integrator and summed THAT, i.e. it
       double-integrated: with a_out = exp(-1/20) ~ 0.951 each timestep's contribution was smeared over
       a ~20-step window and then accumulated again, over-weighting late timesteps and compressing the
       class-logit differences the loss depends on. That readout appears in neither the reference
       implementation nor the manuscript's equations (which specify the neuron, not the readout), and
       it affected EVERY model equally -- a plausible cause of the uniformly low accuracies observed."""
    def __init__(self, cell_name, **cell_kw):
        super().__init__()
        self.cell_name = cell_name
        self.layer1 = RecLayer(cell_name, N_IN, HIDDEN, **cell_kw)
        self.layer2 = RecLayer(cell_name, HIDDEN, HIDDEN, **cell_kw)
        self.w_out = nn.Linear(HIDDEN, N_OUT)
    def forward(self, x):
        B, T, _ = x.shape; dev = x.device
        self.layer1.init(B, dev); self.layer2.init(B, dev)
        out = torch.zeros(B, N_OUT, device=dev)
        n_spk = 0.0
        for t in range(T):
            s1 = self.layer1.step(x[:, t])
            s2 = self.layer2.step(s1)
            n_spk = n_spk + s2.detach().sum()
            out = out + self.w_out(s2)     # plain linear projection, summed over time (official scheme)
        return out, n_spk / B

## Training harness (resumable, shared across sessions)
Model/ablation tag → (cell class, kwargs) mapping. `TAG_SPEC` is the single source of truth for what each run trains.

In [ ]:
TAG_SPEC = {
    'LIF':                     ('LIF', {}),
    'ALIF':                    ('ALIF', {}),
    'TCLIF':                   ('TCLIF', {}),
    'DHLIF':                   ('DHLIF', {}),
    'PGLIF':                   ('PGLIF', {}),
    'PGLIF_ablA_alif2':        ('ALIF2', {}),
    'PGLIF_ablB_kappa0':       ('PGLIF', dict(kappa_fixed_zero=True)),
    'PGLIF_ablD_thetajitter':  ('PGLIF', dict(theta_d_jitter=0.2)),
    'PGLIF_ablE_norefrac':     ('PGLIF', dict(tref_p=0)),
    'PGLIF_ablF_dendriteonly': ('PGLIF', dict(dendrite_only=True)),   # v10: feedforward routes to the
        # dendrite ONLY (soma gets no direct I_ff term, only I_rec + the plateau drive). Quantifies the
        # manuscript's claim (Section 6.2) that this routing "acts as an information bottleneck that
        # collapses accuracy" -- that claim currently has no reported number; this tag produces one.
}
print('Ablation (c) [gate removed, gi=1] is a Regime B (online-learning) concept and is not run here;')
print('the gate has no effect on offline BPTT since it only modulates the plasticity rule, not the forward pass.')

FIX_VERSION = 9   # v9: explicit, AMP-independent skip-on-nonfinite-gradient check added (see training
                    # loop above). v8's clamps are KEPT (they successfully bound the forward pass -- confirmed
                    # by a diagnostic run showing max|vd|=max|vs|=V_CLAMP exactly, max|p| well under P_CLAMP,
                    # loss finite, right up to the moment of a purely BACKWARD-pass non-finite gradient
                    # confined to layer1's parameters). v8's actual failure was that GradScaler's built-in
                    # protection against exactly this is a no-op when USE_AMP=False, which it is by design --
                    # so v8 runs had no real protection at all. This changes ONLY the PG-LIF family's training
                    # LOOP behavior (baselines never produce non-finite gradients in any run so far, so the
                    # check never activates for them); BUGGY set stays restricted to the two affected classes.
BUGGY_UNDER_OLD_CLIP = {'PGLIF', 'ALIF2'}   # only these two cell classes are affected; baselines reused
# Note: PGLIF_ablF_dendriteonly uses cell_name='PGLIF', so it IS in BUGGY_UNDER_OLD_CLIP -- correctly forcing
# it to train fresh every time regardless of fix_version, since (being brand new) it has no prior finished
# result to reuse in the first place; this membership only matters for cross-version REUSE, which is moot here.

def evaluate(model, device):
    model.eval(); correct = tot = 0; spk = 0.0; nb = 0
    amp_on = USE_AMP and device == 'cuda'
    with torch.no_grad(), torch.autocast(device_type='cuda', enabled=amp_on):
        for x, y in batches(TE_X, TE_Y, 128, shuffle=False, device=device):
            out, ns = model(x)
            correct += (out.argmax(1) == y).sum().item(); tot += len(y)
            spk += ns.item(); nb += 1
    return correct / tot, spk / nb

def train_one(tag, seed, device):
    res_file = os.path.join(OUT, f'{tag}_s{seed}.json')
    ckpt_file = os.path.join(OUT, f'{tag}_s{seed}_ckpt.pt')
    cell_name, kw = TAG_SPEC[tag]

    # 1) Fully finished already (this folder) -> skip entirely.
    if os.path.exists(res_file):
        r = json.load(open(res_file))
        if r.get('fix_version', 0) >= FIX_VERSION or cell_name not in BUGGY_UNDER_OLD_CLIP:
            print(f'[skip] {tag} seed {seed} already finished ({res_file})'); return r
        print(f'[redo] {tag} seed {seed} finished under an old clip fix version -> retraining.')

    # 2) Same finished result exists in ANOTHER results folder under P1_results -> reuse, don't retrain.
    #    This branch only runs for cell types NOT in BUGGY_UNDER_OLD_CLIP -- i.e. cells that are byte-identical
    #    to earlier versions. For those, ANY finished result (a written *_s{seed}.json) is valid regardless of
    #    its recorded fix_version or how many epochs it ran: early stopping is a legitimate finish, and an
    #    unchanged cell produces identical results across notebook versions. This is what lets a v8 run reuse
    #    the v7 baselines (fix_version=7, early-stopped at 46-74 epochs) instead of needlessly retraining them.
    if cell_name not in BUGGY_UNDER_OLD_CLIP:
        best_cand = None
        for root, _, files in os.walk(OUT_ROOT):
            fn = f'{tag}_s{seed}.json'
            if fn in files and root != OUT:
                cand = json.load(open(os.path.join(root, fn)))
                if 'best_test_acc' in cand and cand.get('history'):   # a genuinely finished run
                    best_cand = cand   # take the most recently walked finished copy
        if best_cand is not None:
            json.dump(best_cand, open(res_file, 'w'), indent=2)
            print(f'[reuse] {tag} seed {seed}: reused an existing finished result (unchanged cell type, '
                  f'no retraining needed).')
            return best_cand

    torch.manual_seed(seed); np.random.seed(seed)
    model = RecSNN(cell_name, **kw).to(device)
    n_par = sum(p.numel() for p in model.parameters())   # count BEFORE compiling
    base_model = model   # handle to the uncompiled module: always save/load state_dict through this so
                          # checkpoints are compile-agnostic (torch.compile prefixes keys with '_orig_mod.')
    if USE_COMPILE and device == 'cuda':
        try:
            model = torch.compile(model)
            print(f'{tag} s{seed}: torch.compile enabled (first batch will be slower - one-time graph compilation).')
        except Exception as e:
            print(f'{tag} s{seed}: torch.compile failed ({e}) - falling back to eager mode.')
    opt = torch.optim.Adam(model.parameters(), lr=LR)   # reverted to v3's confirmed optimizer
    sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=SCHEDULE, gamma=0.1)  # v3's confirmed schedule
    crit = nn.CrossEntropyLoss()
    amp_on = USE_AMP and device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_on)
    start_ep, best, best_spk, hist, since_best = 0, 0.0, 0.0, [], 0

    # 3) Partial checkpoint from an earlier, interrupted session -> resume from the next epoch.
    #    Version-gated exactly like final results: a checkpoint written before the current FIX_VERSION
    #    is only trusted if this cell type could not have hit the pre-fix gradient-clip bug.
    if os.path.exists(ckpt_file):
        ck = torch.load(ckpt_file, map_location=device)
        stale = ck.get('fix_version', 0) < FIX_VERSION and cell_name in BUGGY_UNDER_OLD_CLIP
        if stale:
            print(f'[discard] {tag} seed {seed}: checkpoint predates fix_version {FIX_VERSION} for a '
                  f'cell type the old clip bug could have corrupted -> ignoring it, starting from epoch 0.')
            os.remove(ckpt_file)
        else:
            base_model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
            if 'scaler' in ck: scaler.load_state_dict(ck['scaler'])
            start_ep, best, best_spk, hist, since_best = ck['epoch'] + 1, ck['best'], ck['best_spk'], ck['hist'], ck['since_best']
            print(f'[resume] {tag} seed {seed} resuming from epoch {start_ep} (checkpoint had best {best:.4f}).')

    for ep in range(start_ep, EPOCHS):
        model.train(); t0 = time.time()
        n_batches = 0; n_skipped = 0
        for x, y in batches(TR_X, TR_Y, BATCH, shuffle=True, device=device, drop_last=True):
            opt.zero_grad()
            with torch.autocast(device_type='cuda', enabled=amp_on):
                out, _ = model(x)
                loss = crit(out, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)   # MUST unscale before the check/clip below, or thresholds are meaningless
            # v9: EXPLICIT finiteness check, ALWAYS ACTIVE regardless of AMP. Root cause established by a
            # dedicated collapse-diagnostic run: PG-LIF's backward pass can produce non-finite gradients
            # (confined to layer1's parameters: w_in, w_rec, ap_logit, kappa) even though every forward-pass
            # state stayed within the V_CLAMP/P_CLAMP bounds and the loss itself was finite -- a genuine BPTT
            # exploding-gradient event through layer1's Jacobian over T=250 unrolled steps, NOT a magnitude
            # runaway the clamps could ever have caught. CRITICALLY: the v8 "fix" relied on GradScaler's
            # built-in skip-on-inf/nan, but GradScaler is a no-op passthrough when USE_AMP=False (which it is,
            # by design, since AMP gave negligible speedup) -- so v8's actual training runs had ZERO protection
            # against this event; the corrupted gradient was applied to the weights every time. This check
            # is independent of AMP and always runs.
            all_finite = all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
            if not all_finite:
                n_skipped += 1
                # v10: DISABLE_FINITENESS_GUARD lets a comparison run apply standard clipping alone (no
                # explicit discard) to test whether clipping by itself is sufficient. Default False preserves
                # the exact v9 behavior byte-for-byte. This flag is read fresh every batch, not cached, so a
                # comparison cell can toggle it around a call to train_one() without touching this function.
                if not globals().get('DISABLE_FINITENESS_GUARD', False):
                    opt.zero_grad()   # discard the corrupted gradients cleanly; do NOT step the optimizer
                else:
                    torch.nn.utils.clip_grad_value_(model.parameters(), 1.0)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    scaler.step(opt)   # NOTE: clip does NOT repair non-finite values (IEEE-754: comparisons
                    scaler.update()    # against NaN are false), so this applies the corrupted gradient anyway
                                        # -- this is deliberate: it is exactly the comparison being tested.
            else:
                torch.nn.utils.clip_grad_value_(model.parameters(), 1.0)   # element-wise clip FIRST:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)    # prevents the v2 bug (see history)
                scaler.step(opt)   # no-op wrapper around opt.step() when amp_on=False; real AMP skip-check
                scaler.update()    # (redundant with the check above, but harmless) when amp_on=True
            n_batches += 1
        sch.step()
        _guard_off = globals().get('DISABLE_FINITENESS_GUARD', False)
        if n_batches and n_skipped / n_batches > 0.30:
            if _guard_off:
                print(f'{tag} s{seed} ep{ep:03d}: SEVERE - {n_skipped}/{n_batches} steps had a non-finite '
                      f'gradient and were APPLIED ANYWAY (finiteness guard disabled for this comparison run) '
                      f'-- expect permanent weight corruption; check n_params_nonfinite_at_end in the result.')
            else:
                print(f'{tag} s{seed} ep{ep:03d}: SEVERE - {n_skipped}/{n_batches} optimizer steps skipped this '
                      f'epoch (non-finite gradient) -- this run is likely fundamentally unstable, not experiencing '
                      f'rare events. Consider treating this run as diverged rather than trusting its accuracy.')
        elif n_skipped > 0:
            if _guard_off:
                print(f'{tag} s{seed} ep{ep:03d}: {n_skipped}/{n_batches} steps had a non-finite gradient and '
                      f'were applied anyway (finiteness guard disabled for this comparison run).')
            else:
                print(f'{tag} s{seed} ep{ep:03d}: {n_skipped}/{n_batches} optimizer steps skipped this epoch '
                      f'(non-finite gradient, discarded before it could reach the weights).')
        acc, spk = evaluate(model, device)
        hist.append({'epoch': ep, 'test_acc': acc, 'spikes': spk, 'skipped_steps': n_skipped, 'total_steps': n_batches})
        if acc > best + 1e-4:
            best, best_spk, since_best = acc, spk, 0
        else:
            since_best += 1
        if ep % 5 == 0 or ep == EPOCHS - 1:
            print(f'{tag} s{seed} ep{ep:03d}  acc {acc:.4f} (best {best:.4f})  '
                  f'spk/sample {spk:.0f}  {time.time()-t0:.0f}s')
        # checkpoint every 5 epochs (and on the last one) so an interrupted session loses at most a few epochs
        if ep % 5 == 0 or ep == EPOCHS - 1:
            torch.save({'model': base_model.state_dict(), 'opt': opt.state_dict(), 'sch': sch.state_dict(),
                        'scaler': scaler.state_dict(), 'epoch': ep, 'best': best, 'best_spk': best_spk,
                        'hist': hist, 'since_best': since_best, 'fix_version': FIX_VERSION}, ckpt_file)
        if EARLY_STOP_PATIENCE and since_best >= EARLY_STOP_PATIENCE and ep >= EARLY_STOP_MIN_EPOCH:
            print(f'{tag} s{seed}: no improvement for {EARLY_STOP_PATIENCE} epochs after the last LR '
                  f'drop -> early stop at ep{ep:03d} (best {best:.4f}).')
            break
    total_skipped = sum(h.get('skipped_steps', 0) for h in hist)
    total_steps_all = sum(h.get('total_steps', 0) for h in hist)
    # v10: explicit check for permanent weight corruption. If DISABLE_FINITENESS_GUARD was set, a non-finite
    # gradient may have been applied via opt.step() anyway; for Adam specifically this poisons the first/second
    # moment estimates (adam_m, adam_v), which then stay NaN for that parameter for the rest of training
    # regardless of future finite gradients (Adam's update divides by sqrt(v)+eps). This check reports whether
    # that actually happened, rather than inferring it indirectly from accuracy alone.
    n_nan_params = sum(1 for p in base_model.parameters() if not torch.isfinite(p).all())
    res = {'tag': tag, 'seed': seed, 'best_test_acc': best, 'spikes_per_sample': best_spk,
           'params': n_par, 'history': hist, 'fix_version': FIX_VERSION,
           'total_skipped_steps': total_skipped, 'total_steps': total_steps_all,
           'skip_rate': (total_skipped / total_steps_all) if total_steps_all else 0.0,
           'finiteness_guard_disabled': globals().get('DISABLE_FINITENESS_GUARD', False),
           'n_params_nonfinite_at_end': n_nan_params}
    json.dump(res, open(res_file, 'w'), indent=2)
    if os.path.exists(ckpt_file): os.remove(ckpt_file)   # finished -> checkpoint no longer needed
    return res

## Run this session
Trains/resumes every tag in `RUN_THIS_SESSION` for every seed in `SEEDS`. Safe to interrupt: rerun the cell (or the whole notebook) later and finished pairs are skipped.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu': print('WARNING: no GPU detected - paper mode will be extremely slow. Switch runtime to GPU.')
results = []
for tag in RUN_THIS_SESSION:
    for sd in SEEDS:
        results.append(train_one(tag, sd, device))

## v10: gradient-clipping-only comparison (reviewer-requested)

Tests whether standard gradient clipping alone (no explicit finiteness-discard guard) is sufficient to train the configurations that showed instability under the normal v9/v10 protocol. **Prediction, stated before running:** it should not be sufficient, because `torch.clamp`/clip operations do not repair non-finite values (IEEE-754 defines every comparison against NaN as false), and applying a non-finite gradient to Adam poisons its first/second moment estimates permanently for that parameter. This cell exists to confirm or refute that prediction empirically rather than assert it.

Runs **seed 0 only** for the four configurations that showed any non-zero discard rate in the main sweep (PGLIF, ablA, ablD, ablE) -- a single-seed diagnostic comparison, not a full statistical claim; the point is to check whether clip-only training corrupts to NaN or merely underperforms, not to establish precise accuracy statistics under this alternative. Results are written as separate `*_cliponly_s0.json` files in the same `OUT` folder, so they never overwrite or interfere with the main sweep's results.

In [ ]:
CLIP_ONLY_TARGETS = ['PGLIF', 'PGLIF_ablA_alif2', 'PGLIF_ablD_thetajitter', 'PGLIF_ablE_norefrac']
clip_only_results = {}
for base_tag in CLIP_ONLY_TARGETS:
    comp_tag = base_tag + '_cliponly'
    TAG_SPEC[comp_tag] = TAG_SPEC[base_tag]   # identical model config; only the training-loop guard differs
    BUGGY_UNDER_OLD_CLIP.add('PGLIF')          # (already present) ensures no stale cross-folder reuse
    DISABLE_FINITENESS_GUARD = True
    try:
        r = train_one(comp_tag, 0, device)
    finally:
        DISABLE_FINITENESS_GUARD = False   # always restore, even if training raises
    clip_only_results[base_tag] = r

print()
print(f"{'config':26s} {'guard acc':>10s} {'clip-only acc':>14s} {'clip-only NaN params?':>24s}")
for base_tag in CLIP_ONLY_TARGETS:
    guard_file = os.path.join(OUT, f'{base_tag}_s0.json')
    guard_acc = json.load(open(guard_file))['best_test_acc']*100 if os.path.exists(guard_file) else float('nan')
    co = clip_only_results[base_tag]
    nan_flag = 'YES -- permanently corrupted' if co['n_params_nonfinite_at_end'] > 0 else 'no'
    print(f"{base_tag:26s} {guard_acc:9.2f}% {co['best_test_acc']*100:13.2f}% {nan_flag:>24s}")


## Aggregate: Table 1 and Table 3
Reads **every** JSON present in the shared results folder (not just this session's), so the tables reflect cumulative progress across sessions even if `RUN_THIS_SESSION` was a subset.

In [ ]:
from collections import defaultdict
NON_RESULT_FILES = {'aggregate.json', 'config.json'}   # files this notebook itself writes into OUT,
                                                        # not per-(tag,seed) results -- must be excluded
                                                        # or a re-run of this cell crashes trying to read
                                                        # its own previous summary as if it were a result.
all_files = [f for f in os.listdir(OUT) if f.endswith('.json') and f not in NON_RESULT_FILES]
by_tag = defaultdict(list)
for f in all_files:
    r = json.load(open(os.path.join(OUT, f)))
    if 'tag' not in r:
        print(f'[skip] {f}: not a (tag, seed) result file (no "tag" key) -- ignoring.'); continue
    by_tag[r['tag']].append(r)

def summarize(tags, label):
    print(f'--- {label} ---')
    rows = []
    for tag in tags:
        rs = by_tag.get(tag, [])
        if not rs:
            print(f'{tag:26s}  (no runs yet)'); continue
        accs = np.array([r['best_test_acc'] for r in rs]); spks = np.array([r['spikes_per_sample'] for r in rs])
        n = len(rs)
        print(f"{tag:26s}  acc {accs.mean()*100:5.2f}" + (f' ± {accs.std()*100:.2f}' if n>1 else '     ')
              + f"%  (n={n} seed{'s' if n!=1 else ''})   spikes/sample {spks.mean():.0f}")
        rows.append({'tag': tag, 'acc_mean': accs.mean(), 'acc_std': accs.std(), 'n': n,
                     'spikes': spks.mean(), 'params': rs[0]['params']})
    return rows

t1 = summarize(list(MODELS_TABLE1), 'Table 1: baselines + PG-LIF')
print()
t3 = summarize(list(MODELS_TABLE3), 'Table 3: ablations')
json.dump({'table1': t1, 'table3': t3}, open(os.path.join(OUT, 'aggregate.json'), 'w'), indent=2)

pg = next((r for r in t1 if r['tag'] == 'PGLIF'), None)
ab = next((r for r in t3 if r['tag'] == 'PGLIF_ablB_kappa0'), None)
if pg and ab and pg['n'] >= 1 and ab['n'] >= 1:
    margin = (pg['acc_mean'] - ab['acc_mean']) * 100
    print(f"\nPre-specified Q1 margin (manuscript Section 6): PG-LIF - kappa0 control = {margin:+.2f} pp "
          f"(need >= +3.0 to claim the memory role; n={min(pg['n'], ab['n'])} seeds so far).")

### Next steps
1. Run additional sessions, widening `SEEDS` toward `[0,1,2,3,4]`, until all Table 1 and Table 3 rows show n = 5.
2. Paste the final `aggregate.json` numbers into manuscript Tables 1 and 3, replacing the single-seed quick-mode numbers; update the abstract, Section 7.1 prose, and the Discussion's remaining "single-seed" caveat once n = 5 for the κ = 0 control.
3. If the multi-seed margin reverses the quick-mode single-seed finding (unlikely given 45.7 vs 45.4 was already within noise), the Section 8 framing must revert accordingly, exactly as pre-registered.
4. Remaining after this: sensitivity analysis (P4), reference verification, DH-LIF validation against a published number.